# 04 — Generalization, Source-Cluster Bootstrap, Failure Analysis

**Research question:** How certain are the held-out scores when nearby video frames are correlated, and which error patterns emerge under the four source/temporal evaluation designs?

All results are **recomputed from the active run's per-frame held-out test predictions**. The bootstrap samples entire reconstructed source groups, not individual frames, and recomputes the Macro F1 for each draw. We report percentile 95% intervals. They quantify variability under the sampled source distribution; they do not prove causal effects. `target=1` always denotes `flip`.

In the smoke profile 40 bootstrap replicates are enough to check the code pathway, **not** to estimate scientific confidence intervals. The full profile uses 5,000.

In [ ]:
from pathlib import Path
import sys,json,numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd().resolve();sys.path.insert(0,str(ROOT/'generated'))
import monreader_methods as mr
settings=mr.load_settings(ROOT)
print('Source-cluster bootstrap replicates:',settings['bootstrap_resamples'])

## ResNet18 uncertainty across all four regimes

The difference between a score and its uncertainty matters. This cell loads frozen test predictions from Notebook 03, resamples **source groups with replacement**, recalculates Macro F1 and saves numerical intervals and a reproducible figure.

In [ ]:
bootstrap_rows=[];all_draws={};error_rows=[]
for regime in mr.REGIMES:
    p=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
    r,draws=mr.source_bootstrap(p,seed=settings['seed'],reps=settings['bootstrap_resamples'])
    bootstrap_rows.append(dict(regime=regime,**r));all_draws[regime]=draws
    error_rows.append(dict(regime=regime,**mr.classify_errors(p)))
bootstrap=pd.DataFrame(bootstrap_rows);errors=pd.DataFrame(error_rows)
bootstrap.to_csv(Path(settings['out'])/'bootstrap_resnet18.csv',index=False)
errors.to_csv(Path(settings['out'])/'resnet18_error_summary.csv',index=False)
display(bootstrap);display(errors)

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(4)
lower=bootstrap.point_macro_f1-bootstrap.ci_lower
upper=bootstrap.ci_upper-bootstrap.point_macro_f1
ax.errorbar(x,bootstrap.point_macro_f1,yerr=np.vstack([lower,upper]),fmt='o',capsize=5)
ax.set_xticks(x,bootstrap.regime);ax.set_ylim(0,1.05)
ax.set(title='ResNet18 Held-out Macro F1 with Source-video Bootstrap 95% CI',ylabel='Macro F1')
fig.tight_layout();path=Path(settings['out'])/'figures'/'resnet18_source_video_bootstrap.png';fig.savefig(path,dpi=150);plt.show()

## Paired RGB-versus-grayscale source bootstrap

For each selected regime, join RGB and grayscale predictions on **exact sample IDs**. Within each bootstrap draw, sample the same source groups for both models. The output estimates the grayscale-minus-RGB **held-out Macro F1 difference**, not tuning F1. A confidence interval containing zero does not establish equivalence; it means the observed difference is not resolved by this interval.

In [ ]:
color_rows=[]
for regime in ('O','ST'):
    rgb=pd.read_csv(Path(settings['out'])/'scratch'/regime/'heldout_test_predictions.csv')
    gray=pd.read_csv(Path(settings['out'])/'grayscale'/regime/'heldout_test_predictions.csv')
    r=mr.paired_source_bootstrap(rgb,gray,seed=settings['seed'],reps=settings['bootstrap_resamples'])
    color_rows.append(dict(regime=regime,**r))
color=pd.DataFrame(color_rows);color.to_csv(Path(settings['out'])/'paired_color_bootstrap.csv',index=False)
display(color)

## Failure modes and confidence

False negatives are real `flip` frames predicted `notflip`; false positives are `notflip` frames predicted `flip`. A high-confidence error means a wrong prediction with confidence above 0.90. These quantities are **counted from individual predictions**. The analysis does not assume that T or ST must contain only false negatives.

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(4);a=ax.bar(x-.20,errors.fp,width=.4,label='False positive');b=ax.bar(x+.20,errors.fn,width=.4,label='False negative')
ax.set_xticks(x,errors.regime);ax.set(ylabel='Errors',title='ResNet18 Held-out False Positives and False Negatives');ax.legend()
fig.tight_layout();path=Path(settings['out'])/'figures'/'resnet18_failure_analysis.png';fig.savefig(path,dpi=150);plt.show()
print('Notebook 04 completed without manually supplied performance numbers.')